# 04 - Risk Analytics

**Financial Portfolio Tracker** | Notebook 4 of 5

---

## Objective

Quantify portfolio risk using multiple complementary approaches: Value at Risk (VaR) with three methodologies, Conditional VaR (Expected Shortfall), risk-adjusted return ratios, CAPM decomposition (beta and alpha), and volatility regime analysis.

## Mathematical Framework

### Value at Risk (VaR)

VaR answers the question: *"What is the maximum loss I can expect over a given period at a given confidence level?"*

**1. Parametric (Gaussian) VaR** -- assumes returns follow a normal distribution:

$$VaR_\alpha = -(\mu + z_\alpha \cdot \sigma)$$

where $\mu$ is the mean daily return, $\sigma$ is the standard deviation, and $z_\alpha$ is the standard normal quantile at confidence level $\alpha$ (e.g., $z_{0.95} \approx -1.645$).

**2. Historical VaR** -- non-parametric, uses actual return distribution:

$$VaR_\alpha = -\text{Percentile}(r, (1-\alpha) \times 100)$$

This approach makes no distributional assumptions and naturally captures fat tails.

**3. Monte Carlo VaR** -- simulates future returns using fitted parameters:

$$r_{\text{sim}} \sim N(\mu, \sigma^2) \quad \Rightarrow \quad VaR_\alpha = -\text{Percentile}(r_{\text{sim}}, (1-\alpha) \times 100)$$

### Conditional VaR (Expected Shortfall)

CVaR answers: *"Given that we are in the worst $(1-\alpha)$ scenarios, what is the expected loss?"*

$$CVaR_\alpha = -E[r \mid r \leq -VaR_\alpha]$$

CVaR is always worse (larger) than VaR and is considered a more coherent risk measure because it accounts for tail severity.

### Risk-Adjusted Ratios

- **Sharpe Ratio**: $S = \frac{R_p - R_f}{\sigma_p}$ -- excess return per unit of total risk
- **Sortino Ratio**: $So = \frac{R_p - R_f}{\sigma_d}$ -- excess return per unit of downside risk only
- **Calmar Ratio**: $C = \frac{CAGR}{|MDD|}$ -- return per unit of worst drawdown

### CAPM Decomposition

- **Beta**: $\beta = \frac{Cov(R_p, R_m)}{Var(R_m)}$ -- sensitivity to market movements
- **Jensen's Alpha**: $\alpha = R_p - [R_f + \beta(R_m - R_f)]$ -- excess return beyond what CAPM predicts

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRADING_DAYS = 252
RISK_FREE_RATE = 0.045
CONFIDENCE = 0.95

TICKERS = ["VOO", "VXUS", "VWO", "BND", "VNQ", "GLD"]
BENCHMARK = "SPY"

np.random.seed(42)

In [ ]:
# Load data
returns = pd.read_parquet(PROCESSED_DIR / "returns.parquet")
portfolio_returns = returns["Portfolio"].dropna()
benchmark_returns = returns[BENCHMARK].dropna()

print(f"Loaded {len(portfolio_returns)} daily return observations")
print(f"Period: {portfolio_returns.index.min().date()} to {portfolio_returns.index.max().date()}")

## 1. Value at Risk -- Three Methods

We implement all three VaR methodologies at the 95% confidence level and compare their estimates.

In [ ]:
# --- Parametric VaR ---
mu = portfolio_returns.mean()
sigma = portfolio_returns.std()
z_score = stats.norm.ppf(1 - CONFIDENCE)

var_parametric = -(mu + z_score * sigma)

print("PARAMETRIC VaR (95%)")
print(f"  Daily mean (mu):  {mu:.6f}")
print(f"  Daily std (sigma): {sigma:.6f}")
print(f"  z-score (5%):     {z_score:.4f}")
print(f"  VaR (1-day, 95%): {var_parametric:.4f} ({var_parametric:.2%})")
print(f"  On $100,000:      ${var_parametric * 100_000:,.0f}")

In [ ]:
# --- Historical VaR ---
var_historical = -np.percentile(portfolio_returns, (1 - CONFIDENCE) * 100)

print("HISTORICAL VaR (95%)")
print(f"  5th percentile of actual returns: {-var_historical:.4f}")
print(f"  VaR (1-day, 95%): {var_historical:.4f} ({var_historical:.2%})")
print(f"  On $100,000:      ${var_historical * 100_000:,.0f}")

In [ ]:
# --- Monte Carlo VaR ---
n_simulations = 10_000
rng = np.random.default_rng(42)
simulated_returns = rng.normal(mu, sigma, n_simulations)

var_montecarlo = -np.percentile(simulated_returns, (1 - CONFIDENCE) * 100)

print("MONTE CARLO VaR (95%)")
print(f"  Simulations: {n_simulations:,}")
print(f"  VaR (1-day, 95%): {var_montecarlo:.4f} ({var_montecarlo:.2%})")
print(f"  On $100,000:      ${var_montecarlo * 100_000:,.0f}")

In [ ]:
# --- Conditional VaR (Expected Shortfall) ---
threshold = np.percentile(portfolio_returns, (1 - CONFIDENCE) * 100)
tail_returns = portfolio_returns[portfolio_returns <= threshold]
cvar_value = -tail_returns.mean()

print("CONDITIONAL VaR / EXPECTED SHORTFALL (95%)")
print(f"  Mean of worst {(1 - CONFIDENCE):.0%} returns: {-cvar_value:.4f}")
print(f"  CVaR (1-day, 95%): {cvar_value:.4f} ({cvar_value:.2%})")
print(f"  On $100,000:       ${cvar_value * 100_000:,.0f}")
print(f"  CVaR / VaR ratio:  {cvar_value / var_historical:.2f}x (tail severity)")

### VaR Comparison Chart

Let's compare all three VaR methods side by side and overlay them on the return distribution.

In [ ]:
# VaR comparison bar chart
var_methods = pd.DataFrame({
    "Method": ["Parametric\n(Gaussian)", "Historical", "Monte Carlo", "CVaR\n(Expected Shortfall)"],
    "VaR (%)": [var_parametric * 100, var_historical * 100, var_montecarlo * 100, cvar_value * 100],
    "VaR ($100K)": [var_parametric * 100_000, var_historical * 100_000,
                    var_montecarlo * 100_000, cvar_value * 100_000],
})

fig = px.bar(
    var_methods, x="Method", y="VaR (%)",
    title="Value at Risk Comparison (95% Confidence, 1-Day Horizon)",
    text="VaR (%)",
    color="Method",
    color_discrete_sequence=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"],
    template="plotly_white",
)

fig.update_traces(texttemplate="%{text:.3f}%", textposition="outside")
fig.update_layout(height=450, showlegend=False, yaxis_title="Potential Daily Loss (%)")

fig.show()

var_methods

In [ ]:
# Return distribution with VaR lines
fig = go.Figure()

# Histogram of actual returns
fig.add_trace(go.Histogram(
    x=portfolio_returns * 100,
    nbinsx=100,
    name="Daily Returns",
    marker_color="rgba(31, 119, 180, 0.6)",
    histnorm="probability density",
))

# Normal distribution overlay
x_range = np.linspace(portfolio_returns.min() * 100, portfolio_returns.max() * 100, 200)
normal_pdf = stats.norm.pdf(x_range, mu * 100, sigma * 100)
fig.add_trace(go.Scatter(
    x=x_range, y=normal_pdf,
    name="Normal Distribution",
    line=dict(color="#ff7f0e", width=2, dash="dash"),
))

# VaR vertical lines
for name, val, color in [
    ("Parametric VaR", var_parametric, "#d62728"),
    ("Historical VaR", var_historical, "#9467bd"),
    ("CVaR (ES)", cvar_value, "#8c564b"),
]:
    fig.add_vline(
        x=-val * 100, line_dash="dash", line_color=color, line_width=2,
        annotation_text=f"{name}: {val:.2%}",
        annotation_position="top left" if val == var_parametric else "top right",
        annotation_font_color=color,
    )

fig.update_layout(
    title="Portfolio Return Distribution with VaR & CVaR Lines",
    xaxis_title="Daily Return (%)",
    yaxis_title="Density",
    height=500,
    template="plotly_white",
    legend=dict(yanchor="top", y=0.95, xanchor="right", x=0.95),
)

fig.show()

**Interpretation**: The histogram reveals slight negative skewness and heavier tails than the normal distribution would predict (positive excess kurtosis). This means parametric VaR *underestimates* true tail risk -- the historical and Monte Carlo methods capture this better. CVaR further quantifies the average loss in the worst scenarios.

## 2. Distribution Analysis

Let's formally test whether portfolio returns follow a normal distribution.

In [ ]:
# Distribution statistics
skewness = portfolio_returns.skew()
kurtosis = portfolio_returns.kurtosis()  # Excess kurtosis (normal = 0)
jb_stat, jb_pvalue = stats.jarque_bera(portfolio_returns.dropna())

print("DISTRIBUTION ANALYSIS")
print("=" * 50)
print(f"  Skewness:          {skewness:.4f}  {'(negative = left tail heavier)' if skewness < 0 else '(positive = right tail heavier)'}")
print(f"  Excess Kurtosis:   {kurtosis:.4f}  {'(leptokurtic = fat tails)' if kurtosis > 0 else '(platykurtic = thin tails)'}")
print(f"  Jarque-Bera stat:  {jb_stat:.2f}")
print(f"  Jarque-Bera p-val: {jb_pvalue:.6f}  {'(reject normality)' if jb_pvalue < 0.05 else '(fail to reject normality)'}")

## 3. Risk-Adjusted Ratios

In [ ]:
def compute_risk_ratios(ret, name, rf_rate=RISK_FREE_RATE):
    """Compute Sharpe, Sortino, and Calmar ratios."""
    n_years = len(ret) / TRADING_DAYS
    total_ret = (1 + ret).prod() - 1
    ann_ret = (1 + total_ret) ** (1 / n_years) - 1
    ann_vol = ret.std() * np.sqrt(TRADING_DAYS)
    
    # Sharpe
    sharpe = (ann_ret - rf_rate) / ann_vol if ann_vol > 0 else 0
    
    # Sortino
    excess_daily = ret - rf_rate / TRADING_DAYS
    downside = excess_daily[excess_daily < 0]
    downside_std = np.sqrt((downside ** 2).mean()) * np.sqrt(TRADING_DAYS)
    sortino = (ann_ret - rf_rate) / downside_std if downside_std > 0 else 0
    
    # Calmar
    wealth = (1 + ret).cumprod()
    max_dd = (wealth / wealth.cummax() - 1).min()
    calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0
    
    return {
        "Name": name,
        "Ann. Return": ann_ret,
        "Ann. Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Max Drawdown": max_dd,
        "Calmar Ratio": calmar,
    }

ratios_df = pd.DataFrame([
    compute_risk_ratios(portfolio_returns, "Portfolio"),
    compute_risk_ratios(benchmark_returns, "SPY (Benchmark)"),
])

# Format for display
display_df = ratios_df.copy()
for col in ["Ann. Return", "Ann. Volatility", "Max Drawdown"]:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.2%}")
for col in ["Sharpe Ratio", "Sortino Ratio", "Calmar Ratio"]:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}")

print("=" * 70)
print("RISK-ADJUSTED RETURN RATIOS")
print("=" * 70)
display_df.set_index("Name").T

## 4. Beta & Alpha (CAPM Decomposition)

Beta measures the portfolio's systematic risk -- how much it moves with the market. A beta < 1 means the portfolio is less volatile than the market. Alpha measures the excess return after accounting for market exposure.

$$\beta = \frac{\text{Cov}(R_p, R_m)}{\text{Var}(R_m)}$$

$$\alpha = R_p - [R_f + \beta \cdot (R_m - R_f)]$$

In [ ]:
# Beta calculation
aligned = pd.concat([portfolio_returns, benchmark_returns], axis=1).dropna()
aligned.columns = ["Portfolio", "SPY"]

covariance = np.cov(aligned["Portfolio"], aligned["SPY"])
portfolio_beta = covariance[0, 1] / covariance[1, 1]

# Alpha calculation (annualised)
ann_ret_p = (1 + aligned["Portfolio"]).prod() ** (TRADING_DAYS / len(aligned)) - 1
ann_ret_m = (1 + aligned["SPY"]).prod() ** (TRADING_DAYS / len(aligned)) - 1
portfolio_alpha = ann_ret_p - (RISK_FREE_RATE + portfolio_beta * (ann_ret_m - RISK_FREE_RATE))

print("CAPM DECOMPOSITION")
print("=" * 50)
print(f"  Beta:  {portfolio_beta:.4f}")
print(f"  Alpha: {portfolio_alpha:.4f} ({portfolio_alpha:.2%} annualised)")
print(f"")
print(f"  Interpretation:")
if portfolio_beta < 1:
    print(f"  - Beta < 1: Portfolio is less sensitive to market moves than SPY")
    print(f"    For every 1% market move, portfolio moves ~{portfolio_beta:.2f}%")
if portfolio_alpha > 0:
    print(f"  - Positive alpha: Portfolio outperforms CAPM-expected returns")
else:
    print(f"  - Negative alpha: Underperformance vs CAPM-expected returns")

In [ ]:
# Beta scatter plot with regression line
slope, intercept, r_value, p_value, std_err = stats.linregress(
    aligned["SPY"], aligned["Portfolio"]
)

fig = go.Figure()

# Scatter of daily returns
fig.add_trace(go.Scatter(
    x=aligned["SPY"] * 100, y=aligned["Portfolio"] * 100,
    mode="markers",
    marker=dict(size=3, color="rgba(31, 119, 180, 0.3)"),
    name="Daily Returns",
))

# Regression line
x_line = np.linspace(aligned["SPY"].min() * 100, aligned["SPY"].max() * 100, 100)
y_line = intercept * 100 + slope * x_line
fig.add_trace(go.Scatter(
    x=x_line, y=y_line,
    mode="lines",
    name=f"Beta = {slope:.3f}, R^2 = {r_value**2:.3f}",
    line=dict(color="#d62728", width=2),
))

# 45-degree reference line (beta = 1)
fig.add_trace(go.Scatter(
    x=x_line, y=x_line,
    mode="lines",
    name="Beta = 1 (reference)",
    line=dict(color="gray", width=1, dash="dash"),
))

fig.update_layout(
    title=f"Portfolio vs Market Returns (Beta = {portfolio_beta:.3f})",
    xaxis_title="SPY Daily Return (%)",
    yaxis_title="Portfolio Daily Return (%)",
    height=500,
    template="plotly_white",
    legend=dict(yanchor="top", y=0.95, xanchor="left", x=0.05),
)

fig.show()

## 5. Rolling Volatility & Regime Analysis

Volatility is not constant -- it tends to cluster (ARCH effects). We analyze rolling volatility to identify regimes of high and low risk.

In [ ]:
# Rolling annualised volatility (30-day and 90-day windows)
roll_vol_30 = portfolio_returns.rolling(30).std() * np.sqrt(TRADING_DAYS)
roll_vol_90 = portfolio_returns.rolling(90).std() * np.sqrt(TRADING_DAYS)
roll_vol_30_spy = benchmark_returns.rolling(30).std() * np.sqrt(TRADING_DAYS)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=roll_vol_30.index, y=roll_vol_30 * 100,
    name="Portfolio (30d)",
    line=dict(color="#1f77b4", width=1),
))
fig.add_trace(go.Scatter(
    x=roll_vol_90.index, y=roll_vol_90 * 100,
    name="Portfolio (90d)",
    line=dict(color="#1f77b4", width=2.5),
))
fig.add_trace(go.Scatter(
    x=roll_vol_30_spy.index, y=roll_vol_30_spy * 100,
    name="SPY (30d)",
    line=dict(color="#ff7f0e", width=1, dash="dash"),
))

# Add volatility regime bands
median_vol = roll_vol_90.median() * 100
fig.add_hline(y=median_vol, line_dash="dot", line_color="gray",
              annotation_text=f"Median vol: {median_vol:.1f}%")

fig.update_layout(
    title="Rolling Annualised Volatility",
    yaxis_title="Annualised Volatility (%)",
    height=450,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.show()

In [ ]:
# Volatility regime classification
vol_90 = portfolio_returns.rolling(90).std() * np.sqrt(TRADING_DAYS)
vol_median = vol_90.median()
vol_q75 = vol_90.quantile(0.75)

regimes = pd.cut(
    vol_90.dropna(),
    bins=[0, vol_median, vol_q75, float('inf')],
    labels=["Low Volatility", "Normal", "High Volatility"]
)

# Returns by regime
ret_by_regime = portfolio_returns.groupby(regimes).agg(
    mean_daily=('mean'),
    std_daily=('std'),
    count=('count'),
    sharpe=lambda x: (x.mean() * TRADING_DAYS - RISK_FREE_RATE) / (x.std() * np.sqrt(TRADING_DAYS)) if x.std() > 0 else 0,
)

ret_by_regime["ann_return"] = ret_by_regime["mean_daily"] * TRADING_DAYS
ret_by_regime["ann_vol"] = ret_by_regime["std_daily"] * np.sqrt(TRADING_DAYS)

print("RETURN STATISTICS BY VOLATILITY REGIME")
print("=" * 60)
ret_by_regime[["count", "ann_return", "ann_vol", "sharpe"]].round(4)

## 6. Per-Asset Risk Contribution

In [ ]:
# Risk metrics per asset
WEIGHTS = {"VOO": 0.30, "VXUS": 0.20, "VWO": 0.10, "BND": 0.20, "VNQ": 0.10, "GLD": 0.10}

asset_risk = []
for t in TICKERS:
    r = returns[t].dropna()
    ann_vol = r.std() * np.sqrt(TRADING_DAYS)
    var_95 = -np.percentile(r, 5)
    
    # Beta vs SPY
    a = pd.concat([r, benchmark_returns], axis=1).dropna()
    cov_mat = np.cov(a.iloc[:, 0], a.iloc[:, 1])
    b = cov_mat[0, 1] / cov_mat[1, 1] if cov_mat[1, 1] != 0 else 0
    
    asset_risk.append({
        "Ticker": t,
        "Weight": WEIGHTS[t],
        "Ann. Vol": ann_vol,
        "VaR (95%)": var_95,
        "Beta vs SPY": b,
        "Risk Contrib.": WEIGHTS[t] * ann_vol,
    })

risk_df = pd.DataFrame(asset_risk)
risk_df["Risk Contrib. %"] = risk_df["Risk Contrib."] / risk_df["Risk Contrib."].sum() * 100

print("=" * 70)
print("PER-ASSET RISK METRICS")
print("=" * 70)
risk_df

## Risk Dashboard Summary

In [ ]:
# Comprehensive risk dashboard
initial_value = 100_000

risk_summary = pd.DataFrame([
    {"Metric": "Annualised Volatility", "Value": f"{portfolio_returns.std() * np.sqrt(TRADING_DAYS):.2%}"},
    {"Metric": "Parametric VaR (95%, 1-day)", "Value": f"{var_parametric:.2%} (${var_parametric * initial_value:,.0f})"},
    {"Metric": "Historical VaR (95%, 1-day)", "Value": f"{var_historical:.2%} (${var_historical * initial_value:,.0f})"},
    {"Metric": "Monte Carlo VaR (95%, 1-day)", "Value": f"{var_montecarlo:.2%} (${var_montecarlo * initial_value:,.0f})"},
    {"Metric": "CVaR / Expected Shortfall (95%)", "Value": f"{cvar_value:.2%} (${cvar_value * initial_value:,.0f})"},
    {"Metric": "Sharpe Ratio", "Value": f"{ratios_df.iloc[0]['Sharpe Ratio']:.3f}"},
    {"Metric": "Sortino Ratio", "Value": f"{ratios_df.iloc[0]['Sortino Ratio']:.3f}"},
    {"Metric": "Calmar Ratio", "Value": f"{ratios_df.iloc[0]['Calmar Ratio']:.3f}"},
    {"Metric": "Beta", "Value": f"{portfolio_beta:.4f}"},
    {"Metric": "Jensen's Alpha", "Value": f"{portfolio_alpha:.2%}"},
    {"Metric": "Skewness", "Value": f"{skewness:.4f}"},
    {"Metric": "Excess Kurtosis", "Value": f"{kurtosis:.4f}"},
])

print("=" * 60)
print(f"RISK DASHBOARD (Portfolio: ${initial_value:,})")
print("=" * 60)
risk_summary

## Summary & Interpretation

**Key Risk Findings:**

1. **VaR comparison**: Historical VaR tends to be higher than parametric VaR because actual returns exhibit fat tails. The parametric method underestimates risk by assuming perfect normality -- a well-known limitation. For regulatory or risk management purposes, the more conservative Historical/CVaR estimates are preferable.

2. **Portfolio beta < 1**: The diversified portfolio has lower systematic risk than a pure equity position. Including bonds (BND), gold (GLD), and real estate (VNQ) reduces market sensitivity. This means the portfolio will underperform in strong bull markets but provide protection in downturns.

3. **Non-normal returns**: The Jarque-Bera test confirms that portfolio returns are not normally distributed (negative skew, excess kurtosis). This has practical implications: parametric VaR should not be the sole risk measure, and tail-risk hedging may be warranted.

4. **Volatility clustering**: Rolling volatility plots show clear regime shifts -- periods of calm followed by spikes during market stress events. An adaptive allocation strategy could reduce weights to volatile assets during high-volatility regimes.

**Next step**: Notebook 05 -- Monte Carlo simulation and efficient frontier optimization to explore whether the current allocation is optimal.